# Unit 4 + 5: Trees and Boosting

Today's arc:
**impurity → single tree → bagging (Random Forest) → boosting (CatBoost) → regression**

Every ensemble method today is built on the same base unit: the **decision tree**. What changes is *how* we combine trees:

- **Bagging** (Random Forest): many deep trees trained *in parallel* on bootstrapped samples, then averaged. Reduces variance.
- **Boosting** (CatBoost): many shallow trees trained *sequentially*, each fitting the errors of the previous ensemble. Reduces bias *and* variance.

In [ ]:
# !pip install --upgrade catboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

# Compatibility shim: older catboost calls the deprecated DataFrame.iteritems,
# which pandas 2.0+ removed. Re-add it if missing.
if not hasattr(pd.DataFrame, 'iteritems'):
    pd.DataFrame.iteritems = pd.DataFrame.items

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_squared_error
from sklearn.datasets import fetch_california_housing
from sklearn import tree

from catboost import CatBoostClassifier, CatBoostRegressor

# First step into machine learning beyond classic regression

In [ ]:
# Warm Up
R1 = np.array([5,8,9])
R2 = np.array([1,0,20])

Calculate the a) Gini co-efficient and b) entropy for R1. Then, write a function that takes in a normalized vector and outputs the Gini co-efficent and Entropy.

In [ ]:
# your code
# Normalize R1 and R2 into probability vectors, then define lambda functions
# G(x) for Gini and H(x) for entropy. Apply them to see the values.
R1_norm = R1 / R1.sum()
R2_norm = R2 / R2.sum()

# Decision Trees

Here, we will implement once again the titanic dataset.

In [ ]:
# Load the Titanic dataset
titanic = sns.load_dataset('titanic')

There are missing values for age. Replace the nans with the median age.

In [ ]:
# your code

Drop the following columns: 'who', 'embarked', 'parch', 'fare', 'deck', 'embark_town', 'alive'

In [ ]:
# your code

Using label_encoder, transform the dataset into integer encodings.

In [ ]:
# Convert categorical columns to numeric using Label Encoding
label_encoder = LabelEncoder()

titanic['sex'] = label_encoder.fit_transform(titanic['sex'])  # Male: 1, Female: 0
titanic['class'] = label_encoder.fit_transform(titanic['class'])  # 1st, 2nd, 3rd -> 0, 1, 2
titanic['alone'] = label_encoder.fit_transform(titanic['alone'])

# Set-up your training by specifying X and y

Use train_test_split to split the data and train.

In [ ]:
# Define features (X) and target (y)
X = # your code
y = # your code

# Split the data into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = # your code

In [ ]:
# Create and train the Decision Tree Classifier
decision_tree = DecisionTreeClassifier(criterion='gini',
                                       max_depth=3)
decision_tree.fit(X_train, y_train)

# Make predictions on the test set
y_pred_dt = decision_tree.predict(X_test)

In [ ]:
feature_names = X.columns

In [ ]:
plt.figure(figsize=(20,10))
tree.plot_tree(decision_tree,
                   feature_names=feature_names,
                   class_names=["Survived", "Died"],
                   filled=True, max_depth=3, fontsize=20,
      )
print("")

### Evaluating your results
Now that you have your predicted values and actual values, plot a scatter plot and give the R2. What do you see?

In [ ]:
sns.regplot(x=y_test, y=y_pred_dt)

Calculate the accuracy, recall, and precision. Then, use the function confusion_matrix and sns.heatmap to plot a confusion matrix.

In [ ]:
# Evaluate the performance
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print(f'Decision Tree Accuracy: {accuracy_dt:.2f}')
print("Classification Report:\n", classification_report(y_test, y_pred_dt))

# Plot confusion matrix
confusion_dt = confusion_matrix(y_test, y_pred_dt)

sns.heatmap(confusion_dt, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'])

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Change the cost criterion from Gini to entropy
Does this change the performance metric?

In [ ]:
# Your code
# Fit a DecisionTreeClassifier with criterion='entropy', predict, and print accuracy + classification report.

### Hyperparameter tuning: How does performance change with increased max depth?
Write code that increases the max depth from 1 to 5, then plot on one figure how this impacts the accuracy.

In [ ]:
# your code
# Loop over max_depth from 1 to 9. For each depth, fit a DecisionTreeClassifier
# and record the accuracy on the test set in a `results` list.

In [ ]:
# your code — plot depth vs results

# Random Forest

Use the function RandomForestClassifier to train a classifier.

In [ ]:
%%time
# Your Code
# Fit a RandomForestClassifier(n_estimators=100, random_state=42),
# then evaluate with accuracy + classification report + confusion matrix.

How does this compare to the decision tree?

## Choose 10 values for n_estimators and 10 values for max_depth
#### Plot a heatmap that shows the grid search on these values

In [ ]:
# your code
# Define train_one_sample(p) that takes a tuple (n_estimators, max_depth),
# fits a RandomForestClassifier, and returns test accuracy.

def train_one_sample(p):
    pass

In [ ]:
trees = np.arange(0,101,5)[1:]
depth = np.arange(0,11,1)[1:]
params = list( product(trees, depth))

In [ ]:
%%time
# your code — loop over params, call train_one_sample, collect results
acc_results = []

In [ ]:
# your code
# Build a DataFrame from params + acc_results, pivot on n_estimators × depth,
# and plot as a heatmap with sns.heatmap.

---
# Boosting: CatBoost

Where bagging trains trees in **parallel** on random subsets, boosting trains them **sequentially** — each new tree tries to correct the mistakes of the previous ensemble.

Why CatBoost specifically:
- Handles categorical features natively (no one-hot encoding needed)
- Strong defaults, minimal tuning
- Other popular options: **XGBoost**, **LightGBM**

Key hyperparameters:
- `iterations` — max number of boosting rounds (trees)
- `learning_rate` — how much each new tree contributes
- `depth` — depth of each individual tree

## Same features (as with DT and RF)

Reload Titanic — this time keep categoricals as **strings** so CatBoost can handle them natively via `cat_features`.

In [ ]:
titanic = sns.load_dataset('titanic')
titanic['age'] = titanic['age'].fillna(titanic['age'].median())
titanic = titanic.drop(columns=['who', 'embarked', 'parch', 'fare',
                                'deck', 'embark_town', 'alive', 'alone'])
titanic.head()

In [ ]:
X = titanic.drop(columns=['survived'])
y = titanic['survived']

cat_features = ['sex', 'class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# your code
# Fit a CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, verbose=100)
# on X_train, y_train. Pass cat_features so CatBoost handles the string columns natively.

catboost_model = 

In [ ]:
# your code — predict on X_test and print accuracy + classification_report

## Other features

Try adding other categorical features. Does your accuracy improve?

In [ ]:
titanic = sns.load_dataset('titanic')
titanic['age'] = titanic['age'].fillna(titanic['age'].median())
titanic = titanic.drop(columns=['who', 'parch', 'alive', 'alone', 'deck', 'embarked'])

In [ ]:
X = titanic.drop(columns=['survived'])
y = titanic['survived']

cat_features = ['sex', 'class', 'embark_town']

# CatBoost doesn't allow NaN in categorical columns — fill with a placeholder string
X[cat_features] = X[cat_features].astype(str).fillna('MISSING')
X[cat_features] = X[cat_features].replace('nan', 'MISSING')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# your code — fit the same CatBoostClassifier with the expanded cat_features
catboost_model = 

In [ ]:
# your code — predict + print metrics

## Compare with Decision Tree and Random Forest

CatBoost handles categoricals natively, but `DecisionTreeClassifier` and `RandomForestClassifier` don't — they need numeric input. One-hot encode the categorical columns with `pd.get_dummies()` and fit both models on the same train/test split.

In [ ]:
# One-hot encode categoricals so sklearn trees can use them
X_enc = pd.get_dummies(X, columns=cat_features)
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_enc, y, test_size=0.2, random_state=42
)

# your code
# 1. Fit DecisionTreeClassifier(random_state=42, max_depth=6)
# 2. Fit RandomForestClassifier(n_estimators=100, random_state=42)
# 3. Print accuracy for DT, RF, and CatBoost (already trained above)

---
# Regression

### Now, try doing regression instead of classification.

Using the California housing dataset, train regression models using Decision Trees, Random Forest, and CatBoost. Then plot them using `sns.regplot`. If the points are annoying, set `scatter=False`.

In [ ]:
# Load the California housing dataset
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='MedHouseVal')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# CatBoost regressor
model = CatBoostRegressor(iterations=4000, learning_rate=0.1, depth=6, verbose=100)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"CatBoost MSE: {mse:.4f}")

In [ ]:
# your code — fit a DecisionTreeRegressor(random_state=42, max_depth=4) and predict

In [ ]:
# your code — fit a RandomForestRegressor(n_estimators=100, random_state=42) and predict

In [ ]:
print("Decision Tree: ", np.corrcoef(y_test, y_pred_dt)[0, 1])
print("Random Forest: ", np.corrcoef(y_test, y_pred_rf)[0, 1])
print("CatBoost:      ", np.corrcoef(y_test, y_pred)[0, 1])

In [ ]:
sns.regplot(x=y_test, y=y_pred_dt, scatter=False, label='Decision Tree', color='gray')
sns.regplot(x=y_test, y=y_pred_rf, scatter=False, label='Random Forest', color='steelblue')
sns.regplot(x=y_test, y=y_pred,    scatter=False, label='CatBoost',      color='indianred')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.tight_layout()